# Predict Customer Churn — RealMLP on Apple M4

RealMLP-style MLP using **TensorFlow + Metal** for `playground-series-s6e3`.  
Optimized for Apple M4 Mac Mini — uses Metal GPU via `tensorflow-metal`.

Pipeline:
1. Install deps + Metal setup
2. Load & preprocess data
3. Feature engineering
4. Optuna tuning (30 trials x 3-fold CV)
5. Multi-seed ensemble (5 seeds x 10 folds = 50 models)
6. Save submission + OOF

In [1]:
# ── Cell 1: Install extras ────────────────────────────────────────────────────
# tensorflow-metal provides GPU acceleration on Apple Silicon.
# If already installed, pip will skip.
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'tensorflow', 'tensorflow-metal', 'optuna', 'tqdm'])
print('Dependencies installed.')

Dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.1.3 which is incompatible.
gradio 5.44.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, but you have peft 0.18.1 which is incompatible.
autogluon-timeseries 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.10.0 which is incompatible.
surya-ocr 0.16.5 requires opencv-python-headless==4.11.0.86, but you have opencv-python-headless 4.13.0.92 which is incompatible.
surya-ocr 0.16.5 requires transformers<4.54.0,>=4.51.2, but you have transformers 4.57.6 which is incompatible.
fastai 2.8.3 requires torch<2.9,>=1.10, but you have torch 2.10.0 which is incompatible.
autogluon-multimodal 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.10.0 which is incompatible

In [2]:
# ── Cell 2: Imports + device detection ────────────────────────────────────────
import os, gc, warnings, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers, callbacks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import optuna
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Device detection ──────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    # Allow memory growth so TF doesn't grab all Metal GPU memory at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE = f'Metal GPU ({len(gpus)})'
    N_DEVICES = len(gpus)
else:
    DEVICE = 'CPU'
    N_DEVICES = 1

DATA_DIR = '../data/'

print(f'TF version  : {tf.__version__}')
print(f'Device      : {DEVICE}')
print(f'GPUs        : {[g.name for g in gpus]}')
print(f'Data dir    : {DATA_DIR}')

TF version  : 2.19.0
Device      : Metal GPU (1)
GPUs        : ['/physical_device:GPU:0']
Data dir    : ../data/


In [3]:
# ── Cell 3: Experiment settings ───────────────────────────────────────────────
RUN_TUNING   = True     # False → skip Optuna, use PRECOMPUTED_PARAMS
N_TRIALS     = 30       # Optuna trials
N_CV_FOLDS   = 3        # Folds inside each Optuna trial (3 for speed)
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10       # Folds per seed in ensemble
TOTAL_MODELS = len(SEEDS) * N_SPLITS
MAX_EPOCHS   = 200
PATIENCE     = 15

# M4 Mac Mini — single Metal GPU, moderate batch size
BATCH_SIZE = 512

PRECOMPUTED_PARAMS = {
    'n_blocks'    : 3,
    'd_block'     : 256,
    'dropout'     : 0.15,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,
}

print(f'Models : {len(SEEDS)} seeds x {N_SPLITS} folds = {TOTAL_MODELS}')
print(f'Tuning : {"ON" if RUN_TUNING else "OFF"} ({N_TRIALS} trials x {N_CV_FOLDS}-fold)')
print(f'Batch  : {BATCH_SIZE}')

Models : 5 seeds x 10 folds = 50
Tuning : ON (30 trials x 3-fold)
Batch  : 512


## 1. Load & Preprocess Data

In [4]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f'Train : {train_df.shape}   Test : {test_df.shape}')

TARGET = 'Churn'

# Robust target encoding — handles strings, ints, mixed types
train_df[TARGET] = train_df[TARGET].astype(str).str.strip().str.lower()
train_df[TARGET] = train_df[TARGET].map({'yes': 1, 'no': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0})
train_df[TARGET] = train_df[TARGET].astype(np.float32)
print(f'Target dist : {train_df[TARGET].value_counts().to_dict()}')

# Joint label encoding
train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
categorical_features = []

for col in features:
    if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
        converted = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
        if converted.notna().sum() > len(df) * 0.5:
            df[col] = converted
        else:
            categorical_features.append(col)
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].fillna('Missing').astype(str))

num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
    df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

print(f'Categoricals : {categorical_features}')
print(f'Numerics     : {num_features}')

Loading data from ../data/ ...
Train : (594194, 21)   Test : (254655, 20)
Target dist : {0.0: 460377, 1.0: 133817}
Categoricals : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerics     : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


## 2. Feature Engineering

In [5]:
def engineer_features(df):
    """Row-level stats + domain interactions on the three numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)
    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']  = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio'] = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)
    return df

X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

X      = pd.get_dummies(X,      columns=categorical_features, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_features, dtype=float)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

D_IN = X.shape[1]
print(f'Train : {X.shape}  |  Test : {X_test.shape}  |  d_in = {D_IN}')
print(f'Target dist : {y.value_counts().to_dict()}')

Train : (594194, 53)  |  Test : (254655, 53)  |  d_in = 53
Target dist : {0.0: 460377, 1.0: 133817}


## 3. Model & Training Utilities (Metal GPU)

In [6]:
X_np      = X.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_np      = y.values.astype(np.float32)

# Track model count for periodic hard reset
_model_count = [0]
_RESET_EVERY = 5  # full keras reset every N models to prevent Metal memory leak


def build_model(d_in, params):
    """Build RealMLP-style model: LayerNorm -> Dense(ReLU) -> Dropout blocks."""
    inputs = layers.Input(shape=(d_in,))
    x = inputs
    for _ in range(params['n_blocks']):
        x = layers.LayerNormalization()(x)
        x = layers.Dense(params['d_block'], activation='relu')(x)
        x = layers.Dropout(params['dropout'])(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=params['lr'],
            weight_decay=params['weight_decay'],
        ),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')],
    )
    return model


def train_one_fold(
    X_tr, y_tr, X_val, y_val, X_tst,
    params,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
):
    """Train one fold on Metal GPU with aggressive memory cleanup."""
    # Periodic hard reset to prevent Metal memory leak / hang
    _model_count[0] += 1
    if _model_count[0] % _RESET_EVERY == 0:
        keras.backend.clear_session()
        gc.collect()
        time.sleep(0.5)  # let Metal release buffers

    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_tst_s = scaler.transform(X_tst)

    model = build_model(X_tr_s.shape[1], params)

    es = callbacks.EarlyStopping(
        monitor='val_auc', mode='max',
        patience=patience, restore_best_weights=True,
    )

    model.fit(
        X_tr_s, y_tr,
        validation_data=(X_val_s, y_val),
        epochs=max_epochs,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )

    val_probs  = model.predict(X_val_s, batch_size=BATCH_SIZE, verbose=0).ravel()
    test_probs = model.predict(X_tst_s, batch_size=BATCH_SIZE, verbose=0).ravel()
    best_auc   = roc_auc_score(y_val, val_probs)

    # Aggressive cleanup for Metal GPU
    del model, X_tr_s, X_val_s, X_tst_s
    keras.backend.clear_session()
    gc.collect()

    return val_probs, test_probs, best_auc


# Warm-up: compile one forward pass so Metal shader compilation doesn't
# slow down the first Optuna trial
print('Warming up (Metal shader compilation) ...')
t0 = time.time()
_m = build_model(D_IN, PRECOMPUTED_PARAMS)
_m.predict(X_test_np[:64], batch_size=64, verbose=0)
del _m; keras.backend.clear_session(); gc.collect()
print(f'Warm-up done in {time.time()-t0:.1f}s — train_one_fold() ready.')

Warming up (Metal shader compilation) ...


2026-03-30 19:45:03.352398: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-30 19:45:03.352435: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-30 19:45:03.352438: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1774880103.352468 9515287 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1774880103.352506 9515287 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-30 19:45:04.075933: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Warm-up done in 1.4s — train_one_fold() ready.


## 4. Optuna Hyperparameter Tuning

30 trials x 3-fold CV. Set `RUN_TUNING = False` in cell 3 to skip and use `PRECOMPUTED_PARAMS`.

In [ ]:
if RUN_TUNING:
    print(f'Optuna: {N_TRIALS} trials x {N_CV_FOLDS}-fold CV  |  device={DEVICE}')
    print(f'Batch : {BATCH_SIZE}\n')

    pbar = tqdm(total=N_TRIALS, desc='Optuna', unit='trial')
    best_so_far = [0.0]

    def objective(trial):
        params = {
            'n_blocks'    : trial.suggest_int('n_blocks', 1, 5),
            'd_block'     : trial.suggest_categorical('d_block', [128, 256, 384, 512]),
            'dropout'     : trial.suggest_float('dropout', 0.0, 0.5),
            'lr'          : trial.suggest_float('lr', 1e-5, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
        }

        skf  = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
        aucs = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np)):
            _, _, auc = train_one_fold(
                X_np[tr_idx], y_np[tr_idx],
                X_np[val_idx], y_np[val_idx],
                X_test_np, params,
                max_epochs=80, patience=8,
            )
            aucs.append(auc)
            trial.report(np.mean(aucs), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        score = np.mean(aucs)
        if score > best_so_far[0]:
            best_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_params = study.best_trial.params
    print(f'\nBest CV AUC : {study.best_value:.5f}')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

    study.trials_dataframe().to_csv('realmlp_m4_tuning_results.csv', index=False)
    print('Tuning results saved to realmlp_m4_tuning_results.csv')

else:
    best_params = PRECOMPUTED_PARAMS
    print('Using PRECOMPUTED_PARAMS:')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

Optuna: 30 trials x 3-fold CV  |  device=Metal GPU (1)
Batch : 512



Optuna:   0%|          | 0/30 [00:00<?, ?trial/s]

## 5. Multi-Seed Ensemble

5 seeds x 10 folds = **50 models** averaged for stable predictions.

In [ ]:
print(f'Ensemble: {len(SEEDS)} seeds x {N_SPLITS} folds = {TOTAL_MODELS} models  |  device={DEVICE}')
print(f'Batch  : {BATCH_SIZE}\n')
t_start = time.time()

test_preds_sum = np.zeros(len(X_test_np))
oof_sum        = np.zeros(len(X_np))
fold_auc_log   = []

for seed in tqdm(SEEDS, desc='Seeds'):
    np.random.seed(seed)
    tf.random.set_seed(seed)

    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X_np))

    for fold, (tr_idx, val_idx) in enumerate(
        tqdm(skf.split(X_np, y_np), total=N_SPLITS, desc=f'  Seed {seed}', leave=False)
    ):
        val_preds, test_preds, fold_auc = train_one_fold(
            X_np[tr_idx], y_np[tr_idx],
            X_np[val_idx], y_np[val_idx],
            X_test_np, best_params,
        )
        fold_auc_log.append(fold_auc)
        seed_oof[val_idx] = val_preds
        oof_sum[val_idx] += val_preds
        test_preds_sum   += test_preds

    seed_auc = roc_auc_score(y_np, seed_oof)
    print(f'Seed {seed}  OOF AUC: {seed_auc:.5f}')

global_oof = oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y_np, global_oof)
elapsed    = time.time() - t_start

print(f'\n{"="*55}')
print(f'Fold AUC  mean={np.mean(fold_auc_log):.5f}  std={np.std(fold_auc_log):.5f}')
print(f'Global Ensemble OOF AUC : {final_auc:.5f}')
print(f'Ensemble time           : {elapsed/60:.1f} min')
print(f'{"="*55}')

## 6. Save Submission & OOF

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET]      = final_test_preds
sub_file         = 'submission_realmlp_m4.csv'
sub.to_csv(sub_file, index=False)
print(f'Submission  : {sub_file}')
print(f'Pred range  : [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]')

oof_df = pd.DataFrame({'id': train_df['id'], 'oof_pred': global_oof})
oof_df.to_csv('oof_realmlp_m4.csv', index=False)
print(f'OOF         : oof_realmlp_m4.csv  (AUC={final_auc:.5f})')

sub.head()

## 7. Submit to Kaggle

In [ ]:
import subprocess

COMPETITION = 'playground-series-s6e3'
SUBMIT_FILE = sub_file
SUBMIT_MSG  = f'RealMLP M4 {TOTAL_MODELS}-model ensemble, Optuna tuned, FE  |  OOF AUC={final_auc:.5f}'
AUTO_SUBMIT = False

print(f'File    : {SUBMIT_FILE}')
print(f'Message : {SUBMIT_MSG}')

if AUTO_SUBMIT:
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', COMPETITION, '-f', SUBMIT_FILE, '-m', SUBMIT_MSG],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    print(f'\nRun manually:')
    print(f'  kaggle competitions submit -c {COMPETITION} -f {SUBMIT_FILE} -m "{SUBMIT_MSG}"')

### To blend with XGBoost / LightGBM
Add to `notebooks/08_blend_submissions.ipynb`:
```python
{'file': 'submission_realmlp_m4.csv', 'label': 'RealMLP-M4', 'public_score': 0.0},
```